In [ ]:
!pip install -q -U kaggle-environments

In [ ]:
import os
import sys
import math
import statistics
import shutil
from pathlib import Path

from kaggle_environments import make

print("Python:", sys.version)
print("Working directory:", os.getcwd())

In [ ]:
%%writefile main.py

# ============================================================
# KAGGRICULTURE SUBMISSION AGENT
# ============================================================

import math

STATE = {}
SEASON_DAYS = 30
ENDGAME_START_DAY = 26

CROPS = {
    "WHEAT": {
        "seed": 10,
        "first_yield_day": 2,
        "max_yield_day": 4,
        "max_yield": 6,
        "ongoing": False,
        "interval": None,
    },
    "CARROT": {
        "seed": 20,
        "first_yield_day": 2,
        "max_yield_day": 3,
        "max_yield": 4,
        "ongoing": False,
        "interval": None,
    },
    "TOMATO": {
        "seed": 50,
        "first_yield_day": 8,
        "max_yield_day": 8,
        "max_yield": 4,
        "ongoing": True,
        "interval": 1,
    },
    "STRAWBERRY": {
        "seed": 100,
        "first_yield_day": 10,
        "max_yield_day": 10,
        "max_yield": 4,
        "ongoing": True,
        "interval": 2,
    },
    "MELON": {
        "seed": 80,
        "first_yield_day": 10,
        "max_yield_day": 12,
        "max_yield": 6,
        "ongoing": False,
        "interval": None,
    },
}

ANIMALS = {
    "GOOSE": {
        "cost": 300,
        "structure": "COOP",
        "product": "EGG",
        "first_yield_day": 4,
        "interval": 1,
    },
    "COW": {
        "cost": 400,
        "structure": "PASTURE",
        "product": "MILK",
        "first_yield_day": 8,
        "interval": 2,
    },
    "SHEEP": {
        "cost": 500,
        "structure": "PASTURE",
        "product": "WOOL",
        "first_yield_day": 6,
        "interval": 3,
    },
}

LAND_PRICES = [1000, 2000, 4000]
LAND_ORDER = ["NE", "SW", "SE"]
MAX_QUADRANTS_TO_BUY = 3
TILES_PER_UNIT = 8

SELL_THROTTLE = {
    "WHEAT": 999,
    "EGG": 999,
    "CARROT": 60,
    "TOMATO": 30,
    "FERTILIZER": 30,
    "MELON": 8,
    "STRAWBERRY": 8,
    "MILK": 8,
    "WOOL": 6,
}


def _shed_tiles(board_size):
    half = board_size // 2
    return [
        (half - 1, half - 1),
        (half, half - 1),
        (half - 1, half),
        (half, half),
    ]


def _step_towards(pos, target):
    fx, fy = pos
    tx, ty = target

    dx = tx - fx
    dy = ty - fy

    if dx == 0 and dy == 0:
        return "PASS"

    if abs(dx) >= abs(dy):
        return "EAST" if dx > 0 else "WEST"

    return "SOUTH" if dy > 0 else "NORTH"


def _dist(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def _fib_cost(n):
    a, b = 1, 1
    for _ in range(n):
        a, b = b, a + b
    return a


def _mature_age(crop):
    cd = CROPS[crop]
    return cd["first_yield_day"] if cd["ongoing"] else cd["max_yield_day"]


def crop_value_per_day(crop, market_prices, day):
    cd = CROPS[crop]

    cycle = _mature_age(crop)
    remaining = (SEASON_DAYS - 1) - day

    if cycle + 1 > remaining:
        return None

    price = market_prices.get(crop, 1)

    if cd["ongoing"]:
        n_events = cd["max_yield"]
        span = cycle + (n_events - 1) * cd["interval"]

        if span + 1 > remaining:
            n_events = max(
                1,
                1 + (remaining - 1 - cycle) // cd["interval"],
            )
            span = cycle + max(0, n_events - 1) * cd["interval"]

        total_yield = n_events * 2
        revenue = total_yield * price
        cost = cd["seed"]
        days_used = max(span, 1)

    else:
        total_yield = cd["max_yield"]
        revenue = total_yield * price
        cost = cd["seed"]
        days_used = cycle

    return (revenue - cost) / days_used


def animal_value_per_day(animal, market_prices, day, active_days_left):
    ad = ANIMALS[animal]

    ramp = ad["first_yield_day"]
    remaining = (SEASON_DAYS - 1) - day

    if ramp + ad["interval"] > remaining:
        return None

    product = ad["product"]
    price = market_prices.get(product, 1)

    active = max(1, remaining - ramp)
    n_events = max(1, active // ad["interval"])

    revenue = n_events * price
    cost = ad["cost"]
    days_used = max(remaining, 1)

    return (revenue - cost) / days_used


def rank_candidates(market_prices, day):
    out = []

    for crop in CROPS:
        value = crop_value_per_day(crop, market_prices, day)
        if value is not None:
            out.append((crop, value, False))

    for animal in ANIMALS:
        value = animal_value_per_day(
            animal,
            market_prices,
            day,
            None,
        )

        if value is not None:
            out.append((animal, value, True))

    out.sort(key=lambda x: -x[1])

    return out


def agent(obs):
    player = obs["player"]
    day = obs["day"]
    hour = obs["hour"]

    if day == 0 and hour == 0:
        STATE.clear()

    farm = obs["farms"][player]
    private = obs["private"]

    money = farm["money"]
    tiles = farm["tiles"]

    board_size = len(tiles)

    shed_tiles = _shed_tiles(board_size)
    shed_set = set(shed_tiles)

    shed = private["shed"]
    seeds = private["seeds"]
    inventories = private["inventories"]

    unlocked = set(farm["unlocked_quadrants"])

    market_prices = obs["market"]["prices"]

    endgame = day >= ENDGAME_START_DAY

    market_orders = []

    # ------------------------------------------------------------
    # FARM SCAN
    # ------------------------------------------------------------

    n_coops = 0
    n_coops_occ = 0
    n_pastures = 0
    n_pastures_occ = 0

    n_plants = 0
    n_weeds = 0

    all_tiles = []
    empties = []

    hungry_animals = []
    unfed_cared_animals = []
    fert_ready_animals = []

    empty_coops = []
    empty_pastures = []

    crop_tile_counts = {c: 0 for c in CROPS}
    animal_tile_counts = {a: 0 for a in ANIMALS}

    for y in range(board_size):
        for x in range(board_size):

            t = tiles[y][x]

            if t == "LOCKED":
                continue

            pos = (x, y)

            if pos in shed_set:
                continue

            all_tiles.append(pos)

            if t is None:
                empties.append(pos)

            elif isinstance(t, dict):

                k = t.get("kind")

                if k == "COOP":

                    n_coops += 1

                    if "animal" in t:
                        n_coops_occ += 1
                        animal_tile_counts["GOOSE"] += 1

                        if not t.get("fed_today", False):
                            hungry_animals.append(pos)

                        elif not t.get("cared_today", False):
                            unfed_cared_animals.append(pos)

                        if t.get("fertilizer_available", False):
                            fert_ready_animals.append(pos)

                    else:
                        empty_coops.append(pos)

                elif k == "PASTURE":

                    n_pastures += 1

                    if "animal" in t:
                        n_pastures_occ += 1
                        animal_tile_counts[t["animal"]] += 1

                        if not t.get("fed_today", False):
                            hungry_animals.append(pos)

                        elif not t.get("cared_today", False):
                            unfed_cared_animals.append(pos)

                        if t.get("fertilizer_available", False):
                            fert_ready_animals.append(pos)

                    else:
                        empty_pastures.append(pos)

                elif k == "PLANT":
                    n_plants += 1
                    crop_tile_counts[t["crop"]] += 1

                elif k == "WEED":
                    n_weeds += 1

    n_units = 1 + len(farm["hands"])
    n_owned_tiles = len(all_tiles)

    STATE["last_weed_ratio"] = (
        n_weeds / max(1, n_owned_tiles)
    )

    if hour >= 1:
        STATE["peak_units"] = max(
            STATE.get("peak_units", n_units),
            n_units,
        )

    reference_units = STATE.get(
        "peak_units",
        n_units,
    )

    # ------------------------------------------------------------
    # LABOR
    # ------------------------------------------------------------

    desired_units = max(
        2,
        min(
            math.ceil(n_owned_tiles / TILES_PER_UNIT),
            26,
        ),
    )

    if endgame:
        desired_units = min(
            desired_units,
            max(2, reference_units),
        )

    desired_hands = desired_units - 1

    ranked = rank_candidates(
        market_prices,
        day,
    )

    best_marginal_value = (
        ranked[0][1]
        if ranked
        else 0.0
    )

    # ------------------------------------------------------------
    # CASH
    # ------------------------------------------------------------

    cash = money

    EMERGENCY_FLOOR = 100
    RECOVERY_FLOOR = 500

    in_emergency = money < RECOVERY_FLOOR

    # ------------------------------------------------------------
    # HIRING
    # ------------------------------------------------------------

    maintenance_hands = max(
        0,
        math.ceil(
            (
                n_plants
                + n_coops_occ
                + n_pastures_occ
            ) / 12
        ) - 1,
    )

    hires_today = farm["hires_today"]
    n_hands = len(farm["hands"])

    worker_daily_value = (
        best_marginal_value
        * (TILES_PER_UNIT * 0.6)
    )

    growth_worth_it = best_marginal_value > 0

    effective_desired_hands = max(
        desired_hands if growth_worth_it else 0,
        maintenance_hands,
    )

    effective_desired_hands = min(
        effective_desired_hands,
        desired_hands,
    )

    if hour < 4 and n_hands < effective_desired_hands:

        n_this_call = hires_today

        while (
            n_hands
            + (n_this_call - hires_today)
            < effective_desired_hands
        ):

            c = _fib_cost(n_this_call)

            if (
                cash - c < EMERGENCY_FLOOR
                or len(market_orders) >= 9
            ):
                break

            is_growth_hand = (
                n_this_call + 1
            ) > maintenance_hands

            if (
                is_growth_hand
                and c > worker_daily_value * 1.5
                and n_this_call > 2
            ):
                break

            market_orders.append(["HIRE"])

            cash -= c
            n_this_call += 1

    # ------------------------------------------------------------
    # SELL
    # ------------------------------------------------------------

    for item, have in list(shed.items()):

        if have <= 0 or item in ANIMALS:
            continue

        cap = (
            9999
            if endgame
            else SELL_THROTTLE.get(item, 10)
        )

        n = min(have, cap)

        if n > 0 and len(market_orders) < 10:
            market_orders.append(
                ["SELL", item, n]
            )

    # ------------------------------------------------------------
    # SEEDS
    # ------------------------------------------------------------

    top_crops = [
        name
        for name, value, is_animal in ranked
        if not is_animal
    ][:3]

    for crop in top_crops:

        if (
            day
            > (SEASON_DAYS - 1)
            - _mature_age(crop)
            - 1
        ):
            continue

        have = seeds.get(crop, 0)
        target = min(4 * n_units, 40)

        cost = CROPS[crop]["seed"]

        if (
            have < target
            and cash - cost >= EMERGENCY_FLOOR
            and len(market_orders) < 10
        ):

            n = min(
                target - have,
                10,
            )

            n = min(
                n,
                max(
                    0,
                    int(
                        (cash - EMERGENCY_FLOOR)
                        // cost
                    ),
                ),
            )

            if n > 0:

                market_orders.append(
                    ["BUY_SEED", crop, n]
                )

                cash -= cost * n

    # ------------------------------------------------------------
    # LAND
    # ------------------------------------------------------------

    weed_ratio = STATE.get(
        "last_weed_ratio",
        0.0,
    )

    n_extra = len(unlocked) - 1

    last_land_day = STATE.get(
        "last_land_buy_day",
        -999,
    )

    if (
        not endgame
        and not in_emergency
        and n_extra < MAX_QUADRANTS_TO_BUY
        and day >= 2
        and weed_ratio < 0.15
        and day >= last_land_day + 4
    ):

        cost = LAND_PRICES[n_extra]

        if (
            cash - cost >= 300
            and len(market_orders) < 10
        ):

            market_orders.append(
                ["BUY_LAND"]
            )

            cash -= cost

            STATE["last_land_buy_day"] = day

    # ------------------------------------------------------------
    # ANIMALS
    # ------------------------------------------------------------

    # Keep disabled exactly as in your v5.
    top_animals = set()

    animal_in_shed = {
        a: shed.get(a, 0)
        for a in ANIMALS
    }

    if not endgame and not in_emergency:

        for animal in ANIMALS:

            if animal not in top_animals:
                continue

            structure = ANIMALS[animal]["structure"]

            empty_list = (
                empty_coops
                if structure == "COOP"
                else empty_pastures
            )

            deficit = max(
                0,
                len(empty_list)
                - animal_in_shed[animal],
            )

            cost = ANIMALS[animal]["cost"]

            if (
                deficit > 0
                and len(market_orders) < 10
            ):

                affordable = max(
                    0,
                    int(
                        (cash - 400)
                        // cost
                    ),
                )

                n_buy = min(
                    deficit,
                    3,
                    affordable,
                )

                if n_buy > 0:

                    market_orders.append(
                        [
                            "BUY_ANIMAL",
                            animal,
                            n_buy,
                        ]
                    )

                    cash -= cost * n_buy
                    animal_in_shed[animal] += n_buy

    want_structure_growth = (
        not endgame
        and bool(top_animals)
        and day >= 3
    )

    market_orders = market_orders[:10]

    # ------------------------------------------------------------
    # SPECIALISTS
    # ------------------------------------------------------------

    total_animals_placed = (
        n_coops_occ
        + n_pastures_occ
    )

    if (
        hour == 0
        or "n_specialists" not in STATE
        or STATE.get("roles_day") != day
    ):

        if (
            endgame
            or total_animals_placed < 5
        ):
            n_specialists = 0

        else:
            n_specialists = max(
                1,
                min(
                    math.ceil(
                        total_animals_placed / 6
                    ),
                    desired_units // 3,
                    desired_units - 1,
                ),
            )

        STATE["n_specialists"] = n_specialists
        STATE["roles_day"] = day

    n_specialists = STATE["n_specialists"]

    # ------------------------------------------------------------
    # WORKERS
    # ------------------------------------------------------------

    all_units = [
        (0, tuple(farm["farmer"]))
    ] + [
        (i + 1, tuple(h))
        for i, h in enumerate(farm["hands"])
    ]

    specialist_ids = (
        set(
            range(
                n_units - n_specialists,
                n_units,
            )
        )
        if n_specialists > 0
        else set()
    )

    tender_units = [
        (idx, pos)
        for idx, pos in all_units
        if idx not in specialist_ids
    ]

    specialist_units = [
        (idx, pos)
        for idx, pos in all_units
        if idx in specialist_ids
    ]

    # ------------------------------------------------------------
    # QUADRANT LOCALITY
    # ------------------------------------------------------------

    def _quadrant_key(p, board_size):
        half = board_size // 2
        x, y = p

        return (
            (0 if y < half else 2)
            + (0 if x < half else 1)
        )

    all_tiles.sort(
        key=lambda p: (
            _quadrant_key(p, board_size),
            p[1],
            p[0],
        )
    )

    n_zone_units = max(
        1,
        len(tender_units),
    )

    chunks = [
        []
        for _ in range(n_zone_units)
    ]

    base_size = (
        len(all_tiles)
        // n_zone_units
    )

    remainder = (
        len(all_tiles)
        % n_zone_units
    )

    idx = 0

    for c in range(n_zone_units):

        size = (
            base_size
            + (1 if c < remainder else 0)
        )

        chunks[c] = all_tiles[
            idx:idx + size
        ]

        idx += size

    # ------------------------------------------------------------
    # PLANTING PLAN
    # ------------------------------------------------------------

    tile_budget = (
        len(tender_units)
        * TILES_PER_UNIT
    )

    plantable_slots = max(
        0,
        tile_budget - n_plants,
    )

    structure_budget = (
        max(0, len(tender_units) // 4)
        if want_structure_growth
        else 0
    )

    empties_sorted = sorted(
        empties,
        key=lambda p: (
            min(
                _dist(p, s)
                for s in shed_tiles
            ),
            p[1],
            p[0],
        ),
    )

    plant_plan = {}

    crop_ranked = [
        name
        for name, value, is_animal in ranked
        if not is_animal
    ]

    slot_i = 0
    struct_placed = 0

    for pos in empties_sorted:

        if (
            slot_i >= plantable_slots
            and struct_placed >= structure_budget
        ):
            break

        if (
            struct_placed < structure_budget
            and slot_i % 6 == 5
            and top_animals
        ):

            preferred_animal = next(
                (
                    n
                    for n, v, isa in ranked
                    if isa
                    and n in top_animals
                ),
                None,
            )

            if preferred_animal:

                plant_plan[pos] = ANIMALS[
                    preferred_animal
                ]["structure"]

                struct_placed += 1
                slot_i += 1

                continue

        if slot_i >= plantable_slots:
            continue

        if not crop_ranked:
            break

        choice = crop_ranked[
            slot_i
            % min(
                3,
                len(crop_ranked),
            )
        ]

        if seeds.get(choice, 0) > 0:

            plant_plan[pos] = choice

        elif seeds.get(
            crop_ranked[0],
            0,
        ) > 0:

            plant_plan[pos] = crop_ranked[0]

        else:
            continue

        slot_i += 1

    actions = {}

    # ------------------------------------------------------------
    # ANIMAL SPECIALISTS
    # ------------------------------------------------------------

    for idx, upos in specialist_units:

        x, y = upos

        inv = (
            inventories[idx]
            if idx < len(inventories)
            else {}
        )

        t = (
            tiles[y][x]
            if 0 <= y < board_size
            and 0 <= x < board_size
            else None
        )

        carried_animal = next(
            (
                a
                for a in ANIMALS
                if inv.get(a, 0) > 0
            ),
            None,
        )

        if carried_animal is not None:

            needed_kind = ANIMALS[
                carried_animal
            ]["structure"]

            candidates = [
                p
                for p in all_tiles
                if isinstance(
                    tiles[p[1]][p[0]],
                    dict,
                )
                and tiles[p[1]][p[0]].get(
                    "kind"
                ) == needed_kind
                and "animal"
                not in tiles[p[1]][p[0]]
            ]

            if candidates:

                target = min(
                    candidates,
                    key=lambda p: _dist(
                        upos,
                        p,
                    ),
                )

                actions[idx] = (
                    ["PLACE", carried_animal]
                    if target == upos
                    else [
                        _step_towards(
                            upos,
                            target,
                        )
                    ]
                )

            else:
                actions[idx] = ["PASS"]

            continue

        if (
            inv.get("WHEAT", 0) > 0
            and hungry_animals
        ):

            target = min(
                hungry_animals,
                key=lambda p: _dist(
                    upos,
                    p,
                ),
            )

            if target == upos:

                actions[idx] = ["FEED"]
                hungry_animals.remove(target)

            else:

                actions[idx] = [
                    _step_towards(
                        upos,
                        target,
                    )
                ]

            continue

        if (
            isinstance(t, dict)
            and "animal" in t
        ):

            if t.get("yield_units", 0) > 0:
                actions[idx] = ["HARVEST"]
                continue

            if (
                not t.get(
                    "fed_today",
                    False,
                )
                and inv.get(
                    "WHEAT",
                    0,
                ) > 0
            ):
                actions[idx] = ["FEED"]
                continue

            if not t.get(
                "cared_today",
                False,
            ):
                actions[idx] = ["CARE"]
                continue

            if t.get(
                "fertilizer_available",
                False,
            ):
                actions[idx] = [
                    "COLLECT_FERTILIZER"
                ]
                continue

        if upos in shed_set:

            if (
                hungry_animals
                and shed.get(
                    "WHEAT",
                    0,
                ) > 0
            ):

                actions[idx] = [
                    "PICKUP",
                    "WHEAT",
                    min(
                        len(hungry_animals) + 2,
                        shed.get(
                            "WHEAT",
                            0,
                        ),
                    ),
                ]

                continue

            for animal, empty_list in (
                (
                    "GOOSE",
                    empty_coops,
                ),
                (
                    "COW",
                    empty_pastures,
                ),
                (
                    "SHEEP",
                    empty_pastures,
                ),
            ):

                if (
                    empty_list
                    and shed.get(
                        animal,
                        0,
                    ) > 0
                ):

                    actions[idx] = [
                        "PICKUP",
                        animal,
                        1,
                    ]

                    break

            if idx in actions:
                continue

        if hungry_animals:

            dest = (
                min(
                    shed_tiles,
                    key=lambda p: _dist(
                        upos,
                        p,
                    ),
                )
                if shed.get(
                    "WHEAT",
                    0,
                ) > 0
                else min(
                    hungry_animals,
                    key=lambda p: _dist(
                        upos,
                        p,
                    ),
                )
            )

            actions[idx] = [
                _step_towards(
                    upos,
                    dest,
                )
            ]

            continue

        any_empty_with_stock = any(
            shed.get(a, 0) > 0
            for a in ANIMALS
        )

        if (
            (empty_coops or empty_pastures)
            and any_empty_with_stock
        ):

            actions[idx] = [
                _step_towards(
                    upos,
                    min(
                        shed_tiles,
                        key=lambda p: _dist(
                            upos,
                            p,
                        ),
                    ),
                )
            ]

            continue

        if unfed_cared_animals:

            target = min(
                unfed_cared_animals,
                key=lambda p: _dist(
                    upos,
                    p,
                ),
            )

            if target == upos:

                actions[idx] = ["CARE"]
                unfed_cared_animals.remove(target)

            else:

                actions[idx] = [
                    _step_towards(
                        upos,
                        target,
                    )
                ]

            continue

        if fert_ready_animals:

            target = min(
                fert_ready_animals,
                key=lambda p: _dist(
                    upos,
                    p,
                ),
            )

            if target == upos:

                actions[idx] = [
                    "COLLECT_FERTILIZER"
                ]

                fert_ready_animals.remove(target)

            else:

                actions[idx] = [
                    _step_towards(
                        upos,
                        target,
                    )
                ]

            continue

        actions[idx] = None

    # ------------------------------------------------------------
    # CROP TENDERS
    # ------------------------------------------------------------

    def crop_action_for(idx, upos, zone):

        x, y = upos

        inv = (
            inventories[idx]
            if idx < len(inventories)
            else {}
        )

        t = (
            tiles[y][x]
            if 0 <= y < board_size
            and 0 <= x < board_size
            else None
        )

        if upos in shed_set:

            carry = {
                k2: v
                for k2, v in inv.items()
                if v > 0
                and k2 not in ANIMALS
            }

            if carry:
                return ["DROP"]

        if isinstance(t, dict):

            k = t.get("kind")

            if k == "PLANT":

                cd = CROPS[t["crop"]]

                age = (
                    day
                    - t["planted_day"]
                )

                mature = (
                    cd["max_yield_day"]
                    if not cd["ongoing"]
                    else cd["first_yield_day"]
                )

                if (
                    t.get(
                        "yield_units",
                        0,
                    ) > 0
                    and age >= mature
                ):
                    return ["HARVEST"]

                if not t.get(
                    "watered_today",
                    False,
                ):
                    return ["WATER"]

                if (
                    inv.get(
                        "FERTILIZER",
                        0,
                    ) > 0
                    and t.get(
                        "fertilized_until_day",
                        -1,
                    ) < day
                ):
                    return ["FERTILIZE"]

            elif k == "WEED":
                return ["DIG"]

        elif (
            t is None
            and upos not in shed_set
            and upos in plant_plan
        ):

            choice = plant_plan[upos]

            if choice in (
                "COOP",
                "PASTURE",
            ):

                del plant_plan[upos]

                return [
                    "BUILD_COOP"
                    if choice == "COOP"
                    else "BUILD_PASTURE"
                ]

            if seeds.get(
                choice,
                0,
            ) > 0:

                del plant_plan[upos]

                return [
                    "PLANT",
                    choice,
                ]

            elif (
                crop_ranked
                and seeds.get(
                    crop_ranked[0],
                    0,
                ) > 0
            ):

                del plant_plan[upos]

                return [
                    "PLANT",
                    crop_ranked[0],
                ]

        def scan(zone_tiles):

            best = {
                "water": None,
                "harvest": None,
                "weed": None,
            }

            bestd = {
                k2: 1e9
                for k2 in best
            }

            for zx, zy in zone_tiles:

                zt = tiles[zy][zx]

                if not isinstance(zt, dict):
                    continue

                zk = zt.get("kind")

                d = _dist(
                    upos,
                    (zx, zy),
                )

                if zk == "PLANT":

                    cd = CROPS[
                        zt["crop"]
                    ]

                    age = (
                        day
                        - zt["planted_day"]
                    )

                    zmature = (
                        cd["max_yield_day"]
                        if not cd["ongoing"]
                        else cd["first_yield_day"]
                    )

                    if (
                        zt.get(
                            "yield_units",
                            0,
                        ) > 0
                        and age >= zmature
                    ):

                        if (
                            d
                            < bestd["harvest"]
                        ):
                            bestd["harvest"] = d
                            best["harvest"] = (
                                zx,
                                zy,
                            )

                    elif not zt.get(
                        "watered_today",
                        False,
                    ):

                        if (
                            d
                            < bestd["water"]
                        ):
                            bestd["water"] = d
                            best["water"] = (
                                zx,
                                zy,
                            )

                elif zk == "WEED":

                    if d < bestd["weed"]:
                        bestd["weed"] = d
                        best["weed"] = (
                            zx,
                            zy,
                        )

            return best

        best = scan(zone)

        chosen = None

        for key in (
            "water",
            "harvest",
            "weed",
        ):

            if best[key] is not None:
                chosen = best[key]
                break

        if chosen is None:

            zone_empty = [
                p
                for p in zone
                if tiles[p[1]][p[0]] is None
                and p in plant_plan
            ]

            if zone_empty:

                chosen = min(
                    zone_empty,
                    key=lambda p: _dist(
                        upos,
                        p,
                    ),
                )

            else:

                best_g = scan(all_tiles)

                for key in (
                    "water",
                    "harvest",
                    "weed",
                ):

                    if best_g[key] is not None:
                        chosen = best_g[key]
                        break

                if (
                    chosen is None
                    and plant_plan
                ):

                    chosen = min(
                        plant_plan.keys(),
                        key=lambda p: _dist(
                            upos,
                            p,
                        ),
                    )

        if chosen is not None:

            return [
                _step_towards(
                    upos,
                    chosen,
                )
            ]

        if upos not in shed_set:

            nearest_shed = min(
                shed_tiles,
                key=lambda p: _dist(
                    upos,
                    p,
                ),
            )

            return [
                _step_towards(
                    upos,
                    nearest_shed,
                )
            ]

        return ["PASS"]

    for i, (idx, upos) in enumerate(
        tender_units
    ):

        zone = chunks[
            i % n_zone_units
        ]

        actions[idx] = crop_action_for(
            idx,
            upos,
            zone,
        )

    for idx, upos in specialist_units:

        if actions.get(idx) is None:

            actions[idx] = crop_action_for(
                idx,
                upos,
                all_tiles,
            )

    farmer_action = actions.get(
        0,
        ["PASS"],
    )

    hands_actions = [
        actions.get(
            i + 1,
            ["PASS"],
        )
        for i in range(
            len(farm["hands"])
        )
    ]

    return {
        "farmer": farmer_action,
        "hands": hands_actions,
        "market": market_orders,
    }

In [ ]:
import py_compile

py_compile.compile(
    "main.py",
    doraise=True
)

print("✅ main.py syntax OK")

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location(
    "submission_agent",
    "main.py"
)

submission_agent = importlib.util.module_from_spec(spec)
spec.loader.exec_module(submission_agent)

assert hasattr(
    submission_agent,
    "agent"
), "main.py does not expose agent(obs)"

print("✅ agent(obs) found")

In [ ]:
env = make(
    "kaggriculture",
    configuration={
        "episodeSteps": 720,
        "seed": 1000,
    },
    debug=True,
)

env.run([
    submission_agent.agent,
    "starter",
])

final = env.steps[-1]

agent_reward = final[0]["reward"]
opponent_reward = final[1]["reward"]

print("Agent:", agent_reward)
print("Starter:", opponent_reward)
print("Status:", final[0]["status"])

In [ ]:
import statistics
from kaggle_environments import make


def run_experiment(
    agent_path="main.py",
    opponent="starter",
    n_seeds=20,
    episode_steps=720,
    base_seed=1000,
):

    results = []
    opponent_results = []

    for i in range(n_seeds):

        seed = base_seed + i

        env = make(
            "kaggriculture",
            configuration={
                "episodeSteps": episode_steps,
                "seed": seed,
            },
            debug=True,
        )

        env.run([
            agent_path,
            opponent,
        ])

        final = env.steps[-1]

        r = final[0]["reward"]
        opp_r = final[1]["reward"]
        status = final[0]["status"]

        results.append(r)
        opponent_results.append(opp_r)

        print(
            f"seed={seed:6d}  "
            f"agent=${r:>10,.0f}  "
            f"opponent=${opp_r:>9,.0f}  "
            f"status={status}"
        )

    print()
    print("=" * 60)
    print(f"n      = {len(results)}")
    print(
        f"mean   = ${statistics.mean(results):,.0f}"
    )
    print(
        f"median = ${statistics.median(results):,.0f}"
    )
    print(
        f"best   = ${max(results):,.0f}"
    )
    print(
        f"worst  = ${min(results):,.0f}"
    )

    if len(results) >= 2:
        print(
            f"std    = ${statistics.stdev(results):,.0f}"
        )

    print(
        f"opp mean = "
        f"${statistics.mean(opponent_results):,.0f}"
    )

    wins = sum(
        a > b
        for a, b in zip(
            results,
            opponent_results,
        )
    )

    ties = sum(
        a == b
        for a, b in zip(
            results,
            opponent_results,
        )
    )

    losses = len(results) - wins - ties

    print(
        f"wins={wins}  "
        f"ties={ties}  "
        f"losses={losses}"
    )

    return results, opponent_results

In [ ]:
results, opponent_results = run_experiment(
    agent_path="main.py",
    opponent="starter",
    n_seeds=20,
    episode_steps=720,
    base_seed=1000,
)

In [ ]:
import json

experiment = {
    "agent": "main.py",
    "opponent": "starter",
    "n_seeds": len(results),
    "base_seed": 1000,
    "episode_steps": 720,
    "results": results,
    "opponent_results": opponent_results,
    "mean": statistics.mean(results),
    "median": statistics.median(results),
    "best": max(results),
    "worst": min(results),
    "std": (
        statistics.stdev(results)
        if len(results) > 1
        else 0
    ),
}

with open(
    "experiment_results.json",
    "w",
) as f:
    json.dump(
        experiment,
        f,
        indent=2,
    )

print("✅ Saved experiment_results.json")

In [ ]:
random_results, random_opponent_results = run_experiment(
    agent_path="main.py",
    opponent="random",
    n_seeds=10,
    episode_steps=720,
    base_seed=2000,
)

In [ ]:
mirror_results, mirror_opponent_results = run_experiment(
    agent_path="main.py",
    opponent="main.py",
    n_seeds=10,
    episode_steps=720,
    base_seed=3000,
)

In [ ]:
from pathlib import Path

path = Path("main.py")

print("File:", path)
print("Size:", path.stat().st_size, "bytes")
print()

text = path.read_text()

print("Contains agent():", "def agent(obs):" in text)
print("Contains STATE:", "STATE =" in text)
print("Contains external imports:", [
    line for line in text.splitlines()
    if line.startswith("import ")
    or line.startswith("from ")
])

In [ ]:
FINAL_SEED = 9999

env = make(
    "kaggriculture",
    configuration={
        "episodeSteps": 720,
        "seed": FINAL_SEED,
    },
    debug=True,
)

env.run([
    "main.py",
    "starter",
])

final = env.steps[-1]

print("FINAL VALIDATION")
print("=" * 50)
print("Agent reward   :", final[0]["reward"])
print("Opponent reward:", final[1]["reward"])
print("Agent status   :", final[0]["status"])

In [ ]:
import shutil

shutil.copyfile(
    "main.py",
    "submission.py",
)

print("✅ Final submission copied to submission.py")